# 01: Data lake inventory

| Source | What it gives | Data scanned |
|---|---|---|
| Glue API | every table, its format, location, columns, partition keys | none — not an Athena query |
| `information_schema.columns` | the schema Athena actually resolves | none |
| `"table$partitions"` (Iceberg) | **exact per-partition row counts and byte sizes** | manifests only, ~0 |
| dimension tables | site and circuit counts | a few MB |
| partition-filtered `LIMIT 5` | what a row looks like | one partition, a handful of columns |

## Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from bms_sa_review.synthetic_ami_creation.config import ami_config as Config
from bms_sa_review.synthetic_ami_creation.lib import ami_athena as Athena
from bms_sa_review.synthetic_ami_creation.lib import ami_inventory as Inventory

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

Athena.reset_scan_log()
Athena.require_credentials()
print("Credentials OK. Starting inventory.")


## 1. The catalogue

- `is_iceberg` is the column to read first. 
- An Iceberg table exposes `$partitions`,`$files`, `$snapshots` and `$history`, which is how the next section gets row counts for nothing. A Hive table does not.


In [ ]:
catalog = Inventory.glue_inventory()
print(f"{len(catalog)} tables across {catalog.database.nunique()} databases")
display(Inventory.database_summary(catalog))


In [ ]:
display(catalog[["database", "table", "table_type", "format", "is_iceberg",
                 "n_columns", "partition_keys", "updated"]])


In [ ]:
#### CHECK DATES for ts ####
partitions = Athena.aq("""
    SELECT DISTINCT year, month
    FROM "ts$partitions"
    ORDER BY year, month
""")
print(partitions)

### Storage locations

Where the files physically are. Two tables pointing at the same prefix, or a table
whose location is somewhere unexpected, is worth checking


In [ ]:
display(catalog[["database", "table", "location"]].sort_values(["location"])
        .reset_index(drop=True))


## 2. Real schemas

- `information_schema.columns` is what **Athena** will let you select. 
- That is not always what Glue declares
- `aws_config.describe()` notes that the Glue column list can come back empty for Iceberg tables, which is why `n_columns` above may disagree with what you see here.

One query per database rather than one `DESCRIBE` per table.

In [ ]:
columns = pd.concat(
    [Inventory.column_inventory(db).assign(database=db) for db in (Config.SA, Config.SAI)],
    ignore_index=True,
)
print(f"{len(columns):,} column definitions across {columns.table_name.nunique()} tables")

width = (columns.groupby(["database", "table_name"], as_index=False)
                .agg(n_columns=("column_name", "count")))
display(width.sort_values("n_columns", ascending=False).reset_index(drop=True))


## 3. Partition metadata

- For every table that declares partition keys, read its `$partitions` metadata table. 
- On Iceberg this returns exact record counts, file counts and total bytes per partition, from the manifests, without touching data.
- Tables that will not serve `$partitions` are logged and skipped


In [ ]:
totals, raw_partitions, partition_log = Inventory.probe_partitions(catalog)
print()
display(partition_log)


### What `$partitions` actually returned


In [ ]:
ts_key = next((k for k in raw_partitions if k.endswith(".ts")), None)
if ts_key is None:
    print("No `ts` partition metadata was returned. Available keys:")
    for key in sorted(raw_partitions):
        print("  -", key)
else:
    print(f"raw {ts_key}$partitions -- {len(raw_partitions[ts_key])} rows, "
          f"columns: {list(raw_partitions[ts_key].columns)}")
    display(raw_partitions[ts_key].head(10))


## 4. One-screen summary

In [ ]:
summary = Inventory.summary_table(catalog, totals)
display(summary)


## 5. `ts` in detail + `is_pv`

- `ts` is partitioned on `(year, month, is_pv)`.
- Every query in the Stage 1 pipeline filters `is_pv = True`. 
- If `ts` has no `is_pv = False` partitions, or they are near-empty, then the load circuits a synthetic meter needs do not exist in this table

In [ ]:
ts_tidy = pd.DataFrame()
if ts_key is not None:
    ts_tidy = Inventory.normalise_partitions(raw_partitions[ts_key])
    display(ts_tidy)

    if "is_pv" in ts_tidy.columns and "n_rows" in ts_tidy.columns:
        by_is_pv = (ts_tidy.groupby("is_pv", as_index=False)
                           .agg(n_partitions=("is_pv", "count"),
                                n_rows=("n_rows", "sum"),
                                size_bytes=("size_bytes", "sum")))
        by_is_pv["size"] = by_is_pv.size_bytes.map(Athena.fmt_bytes)
        by_is_pv["share_of_rows"] = (
            by_is_pv.n_rows / by_is_pv.n_rows.sum()
        ).map(lambda v: f"{v:.1%}")
        print("ts rows by is_pv partition:")
        display(by_is_pv[["is_pv", "n_partitions", "n_rows", "size", "share_of_rows"]])
    else:
        print("`$partitions` did not carry an is_pv breakdown with row counts.")
        print("Columns available:", list(ts_tidy.columns))
        print("Phase 2 will have to establish is_pv coverage another way.")


### Coverage over time

Rows per month, so gaps and the usable date range are visible rather than assumed.


In [ ]:
if len(ts_tidy) and {"year", "month"} <= set(ts_tidy.columns):
    by_month = (ts_tidy.dropna(subset=["year", "month"])
                       .groupby(["year", "month"], as_index=False)
                       .agg(n_rows=("n_rows", "sum") if "n_rows" in ts_tidy.columns
                                     else ("month", "count"),
                            n_partitions=("month", "count")))
    by_month["ym"] = (by_month.year.astype(int).astype(str) + "-"
                      + by_month.month.astype(int).astype(str).str.zfill(2))
    display(by_month[["ym", "n_partitions", "n_rows"]])
    print(f"Coverage: {by_month.ym.iloc[0]} .. {by_month.ym.iloc[-1]} "
          f"({len(by_month)} month-partitions)")
else:
    print("No year/month partition metadata available for ts.")


## 6. Candidate tables

The shortlist is a naming heuristic, it does not exclude anything. 


In [ ]:
shortlisted = Inventory.guess_candidates(catalog)
candidates = shortlisted[shortlisted.shortlisted].reset_index(drop=True)
print(f"{len(candidates)} of {len(catalog)} tables shortlisted by name.")
display(candidates[["database", "table", "is_iceberg", "n_columns", "partition_keys"]])


In [ ]:
for entry in candidates.itertuples(index=False):
    key = f"{entry.database}.{entry.table}"
    schema = columns[(columns.database == entry.database)
                     & (columns.table_name == entry.table)]
    stats = totals.get(key, {})

    # Prefer the ACTUAL partition columns $partitions found over Glue's declared
    # PartitionKeys: Iceberg hides its partition spec from Glue, so `ts` shows
    # "(none)" in `entry.partition_keys` despite genuinely being partitioned --
    # see `Inventory.should_probe_partitions`.
    declared = entry.partition_keys
    actual = ", ".join(stats.get("partition_columns", []) or [])
    if declared:
        partition_desc = declared
    elif actual:
        partition_desc = f"{actual}  [not declared in Glue -- recovered from $partitions]"
    else:
        partition_desc = "(none)"

    print("=" * 78)
    print(f"{key}   [{'iceberg' if entry.is_iceberg else 'hive'}]"
          f"   partitioned by: {partition_desc}")
    if stats.get("n_partitions"):
        print(f"  {stats['n_partitions']} partitions"
              + (f", {stats['n_rows']:,.0f} rows" if stats.get("n_rows") else "")
              + (f", {Athena.fmt_bytes(stats['size_bytes'])}" if stats.get("size_bytes") else "")
              + (f", {stats.get('first_partition')} .. {stats.get('last_partition')}"
                 if stats.get("first_partition") else ""))
    if len(schema):
        display(schema[["ordinal_position", "column_name", "data_type"]]
                .reset_index(drop=True))
    else:
        print("  (no columns resolved via information_schema)")


## 7. Sample rows

`meta_up23c` is the circuit dimension. one row per circuit.


In [ ]:
latest_year = latest_month = None
if len(ts_tidy) and {"year", "month"} <= set(ts_tidy.columns):
    ordered = ts_tidy.dropna(subset=["year", "month"]).sort_values(["year", "month"])
    latest_year = int(ordered.year.iloc[-1])
    latest_month = int(ordered.month.iloc[-1])
    print(f"Sampling ts from the newest partition: year={latest_year} month={latest_month}")
else:
    print("No partition metadata for ts -- skipping the ts sample rather than guessing.")


In [ ]:
if latest_year is not None:
    ts_sample = Athena.aq(
        f"""
        SELECT *
        FROM ts
        WHERE year = {latest_year} AND month = {latest_month}
        LIMIT 5
        """,
        database=Config.SAI,
        label=f"ts sample {latest_year}-{latest_month:02d}",
    )
    display(ts_sample)
    print("Columns:", list(ts_sample.columns))


In [ ]:
meta_sample = Athena.aq("SELECT * FROM meta_up23c LIMIT 5", database=Config.SAI,
                   label="meta_up23c LIMIT 5")
display(meta_sample)
print("meta_up23c columns:", list(meta_sample.columns))


In [ ]:
fleet = Inventory.dimension_counts("meta_up23c", database=Config.SAI)
display(fleet)

for table in ("circuits", "sites"):
    try:
        display(Athena.aq(f"SELECT * FROM {table} LIMIT 5", database=Config.SA,
                     label=f"{table} LIMIT 5"))
    except Exception as exc:
        print(f"{table}: {type(exc).__name__}: {str(exc)[:200]}")


In [ ]:
# `partition_lookup` is the pipeline's own record of which partitions exist.
# Worth cross-checking against the $partitions metadata above -- a disagreement
# means one of them is stale.
try:
    lookup = Athena.aq("SELECT * FROM partition_lookup", database=Config.SA,
                  label="partition_lookup")
    print(f"{len(lookup)} rows")
    display(lookup.head(30))
except Exception as exc:
    print(f"partition_lookup: {type(exc).__name__}: {str(exc)[:200]}")


## 8. `structured_data`

`ciccada_config.TABLES` maps the logical name to the rebuilt table. 

`build_structured_data.py`:  aggregates circuits to **site** level and filters `is_pv = True` throughout. 
If that filter has already discarded the load circuits, this table cannot be the source.

In [ ]:
structured = Config.TABLES["structured_data"]
print(f"structured_data -> {structured}"
      f"   (rebuilt: {'structured_data' in Config.REBUILT})")

structured_key = f"{Config.SAI}.{structured}"
if structured_key in totals:
    print(totals[structured_key])

structured_schema = columns[columns.table_name == structured]
display(structured_schema[["database", "ordinal_position", "column_name", "data_type"]]
        .reset_index(drop=True))


## 9. Cost


In [ ]:
display(Athena.scan_report())

## 10. Inventory

### Inventory

- **6 Glue databases, 55 tables.** Only 2 are in scope for this project:
  `solar_analytics_iceberg` (35 tables, all Iceberg -- the primary catalogue) and
  `solar_analytics` (13 tables, legacy Hive). `bom_nci` (satellite irradiance,
  8.88B rows) is available but not needed yet.
- **`ts` (`solar_analytics_iceberg.ts`):** 16,345,254,058 rows, ~456.9 GB
  compressed. Partitioned on `(year, month, is_pv)` -- **hidden from Glue's
  declared PartitionKeys**, a real Iceberg-on-Glue quirk, not a data problem
  (see `Inventory.should_probe_partitions`).
- **The `is_pv` split -- the single most consequential number in this phase:**

  | | rows | share | size |
  |---|---:|---:|---:|
  | `is_pv = false` (load) | 8,521,715,460 | 52.1% | 252.0 GB |
  | `is_pv = true` (PV) | 7,823,538,598 | 47.9% | 204.9 GB |

  Load-circuit readings are not a discarded minority -- they are, in raw `ts`,
  slightly the LARGER half of the table. This directly answers the concern in
  the original brief.
- **Fleet size:** 41,393 sites, 171,411 circuits (`circuits` dimension table;
  ~4.14 circuits/site on average). `meta_up23c` has 423,990 rows -- **2.47x
  more than `circuits`, so it is NOT one row per `circuit_id`.**
  `build_structured_data.py` already guards this with `GROUP BY circuit_id` +
  `max(...)` before joining; that convention carries forward rather than being
  rediscovered in Phase 3.
- **`ts` reports 816 total partitions -- RESOLVED.** Not day-level partition
  evolution (my initial guess, and wrong). The raw `$partitions` struct shows a
  FOURTH partition key beyond `(year, month, is_pv)` -- a postcode-bucket
  dimension (exact name truncated in display; confirm with
  `list(ts_tidy.columns)`). Arithmetic proof: 24 months x 2 `is_pv` values x 17
  buckets = 816, exact, and the by-month row counts sum to exactly the total
  row count. `ts` is partitioned at MONTH granularity throughout -- no day-level
  data, no partition-spec evolution. Matters for Phase 4: chunking by
  `(year, month)` alone undercounts the true partition count 17x.
- **Date coverage, confirmed: 2024-01 .. 2025-12, 24 consecutive months, no
  gaps.** (`ami_config.TS_COVERAGE`.)
- **`structured_data` is confirmed PV-only**, independent of any query result:
  its schema has no `is_pv` column at all (none needed -- `ts.is_pv = True` is
  baked into `build_structured_data.py` before the table is written), and its
  partitioning is `(year, month)` only, consistent with a table that only ever
  holds one `is_pv` value. This applies to all three variants seen in the
  catalogue (`structured_data`, `structured_data_v2`,
  `structured_data_v2_flex_included`).
- Every other candidate returned by the naming heuristic
  (`all_uncurtailedpv*`, `conformance_*`, `pv_ghi_norm_model*`, `split_days*`,
  `lso_*`, `voltwatt_uncurtailedpv`) is a DERIVED analytical output of the
  Stage 1/2 pipeline -- modelled PV, conformance verdicts, GHI model artefacts
  -- not raw signal data. Their legacy/`_v2`/`_v2_flex_included` naming pattern
  matches `ciccada_config.REBUILT` exactly (e.g. `conformance_sust_op_3w` and
  `conformance_antiisland` appear ONLY as bare legacy tables, no `_v2` variant
  -- consistent with `ciccada_config`'s own note that those two were not
  rebuilt), which is a good independent cross-check that this inventory and
  the pipeline's own documentation agree.
